In [68]:
#get database of every gameID after all star break
#get player box scores from every gameID w 3FGA, 3FG%
#get defensive data before all star break
#merge and model
from nba_api.stats.endpoints import boxscoretraditionalv2, boxscoreplayertrackv2, boxscoresummaryv2, playerdashboardbygamesplits, playerdashboardbyshootingsplits, playerdashboardbygeneralsplits, shotchartdetail, scoreboardv2, teamplayeronoffsummary, teamplayeronoffdetails, leaguedashplayershotlocations, defensehub, playergamelog, teamdashlineups, leaguedashptteamdefend, leaguedashptdefend
from nba_api.stats.endpoints import leaguegamefinder, playbyplay, SynergyPlayTypes, playbyplayv2, playerawards, PlayByPlay, PlayByPlayV2, leaguegamelog, winprobabilitypbp, WinProbabilityPBP
import pandas as pd


In [29]:

pd.set_option('display.max_columns', 1000)


gamefinder = leaguegamefinder.LeagueGameFinder()
games = gamefinder.get_data_frames()[0]
games


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22023,1610612766,CHA,Charlotte Hornets,0022301159,2024-04-10,CHA @ ATL,W,241,115,40,79,0.506,14,36,0.389,21,22,0.955,5,28,33,25,11,2,13,18,1.0
1,22023,1610612756,PHX,Phoenix Suns,0022301165,2024-04-10,PHX @ LAC,None,0,2,0,2,0.000,0,2,0.000,2,2,1.000,1,0,1,0,1,0,2,1,-5.0
2,22023,1610612753,ORL,Orlando Magic,0022301162,2024-04-10,ORL @ MIL,L,239,99,36,78,0.462,11,30,0.367,16,23,0.696,5,29,34,19,6,1,15,15,-18.0
3,22023,1610612760,OKC,Oklahoma City Thunder,0022301163,2024-04-10,OKC vs. SAS,W,241,127,47,101,0.465,16,38,0.421,17,21,0.810,7,55,62,33,12,6,12,16,38.0
4,22023,1610612742,DAL,Dallas Mavericks,0022301161,2024-04-10,DAL @ MIA,W,241,111,43,87,0.494,15,44,0.341,10,13,0.769,8,31,39,27,8,8,7,13,19.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,22019,220,PCG,Pacers Gaming,1221900023,2019-04-18,PCG @ PGT,W,120,60,25,41,0.610,9,17,0.529,1,2,0.500,4,9,13,18,7,2,5,7,7.0
29996,42018,1610612744,GSW,Golden State Warriors,0041800143,2019-04-18,GSW @ LAC,W,242,132,51,93,0.548,15,35,0.429,15,18,0.833,10,40,50,35,4,11,12,28,27.0
29997,42018,1610612755,PHI,Philadelphia 76ers,0041800123,2019-04-18,PHI @ BKN,W,240,131,45,94,0.479,11,27,0.407,30,35,0.857,15,39,54,26,9,5,16,29,16.0
29998,22019,229,WGS,Warriors Gaming Squad,1221900016,2019-04-17,WGS @ BZG,L,120,53,21,40,0.525,11,22,0.500,0,0,NaN,9,5,14,16,5,4,10,8,-11.0


In [1]:
gameid = '0022301165'
playerid = '1641705'
teamid = '1610612756'
#dashboard = playerdashboardbygamesplits.PlayerDashboardByGameSplits(player_id="203076")
#print(dashboard.get_data_frames())
#box_trad = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=gameid)
#print(box_trad.get_data_frames()[0])
#gives you player box scores. needs opponent tho and player career 3pt percentage
shotchart = shotchartdetail.ShotChartDetail(context_measure_simple='FGA')
print(shotchart.get_data_frames())
scoreboardv2
teamplayeronoffsummary
teamplayeronoffdetails
leaguedashplayershotlocations
defensehub
playergamelog
teamdashlineups
leaguedashptteamdefend
leaguedashptdefend

NameError: name 'shotchartdetail' is not defined

In [36]:
from nba_api.live.nba.endpoints import scoreboard
# Query nba.live.endpoints.scoreboard and  list games in localTimeZone
from datetime import datetime, timezone
from dateutil import parser

f = "{gameId}: {awayTeam} vs. {homeTeam} @ {gameTimeLTZ}" 

board = scoreboard.ScoreBoard()
games = board.games.get_dict()
for game in games:
    gameTimeLTZ = parser.parse(game["gameTimeUTC"]).replace(tzinfo=timezone.utc).astimezone(tz=None)
    print(f.format(gameId=game['gameId'], awayTeam=game['awayTeam']['teamName'], homeTeam=game['homeTeam']['teamName'], gameTimeLTZ=gameTimeLTZ))


0022301165: Suns vs. Clippers @ 2024-04-10 22:30:00-04:00
0022301164: Timberwolves vs. Nuggets @ 2024-04-10 22:00:00-04:00
0022301158: Grizzlies vs. Cavaliers @ 2024-04-10 19:00:00-04:00
0022301159: Hornets vs. Hawks @ 2024-04-10 19:30:00-04:00
0022301160: Raptors vs. Nets @ 2024-04-10 19:30:00-04:00
0022301161: Mavericks vs. Heat @ 2024-04-10 19:30:00-04:00
0022301162: Magic vs. Bucks @ 2024-04-10 20:00:00-04:00
0022301163: Spurs vs. Thunder @ 2024-04-10 20:00:00-04:00


In [61]:
import pandas as pd

def transform_actions(actions, team_id):
    # Parse 'clock' to seconds remaining in the game
    actions['clock_seconds'] = actions['clock'].str.extract(r'PT(\d+)M(\d+.\d+)S').apply(
        lambda x: int(x[0]) * 60 + float(x[1]), axis=1)

    # Create boolean variables for each unique string in 'actionType'
    action_types = pd.get_dummies(actions['actionType'])
    actions = pd.concat([actions, action_types], axis=1)

    # Define the list of subtypes to create booleans for
    subtypes_of_interest = ['Jump Shot', 'Layup', 'Hook', 'DUNK', 'bad pass', 'out', 'in', 'offensive foul']
    for subtype in subtypes_of_interest:
        actions[f'subtype_{subtype}'] = actions['subType'].apply(lambda x: x == subtype)

    # Create booleans for specified qualifiers
    qualifiers_of_interest = ['2ndchance', 'pointsinthepaint', 'fromturnover', 'fastbreak']
    for qualifier in qualifiers_of_interest:
        actions[f'qualifier_{qualifier}'] = actions['qualifiers'].apply(lambda x: qualifier in str(x))

    # Convert 'scorehome' and 'scoreaway' to numeric
    actions['scorehome'] = pd.to_numeric(actions['scoreHome'])
    actions['scoreaway'] = pd.to_numeric(actions['scoreAway'])

    # Create a boolean column to indicate if the teamId matches
    actions['is_team_action'] = actions['teamId'] == team_id

    # Create boolean variables for 'side's unique values
    side_booleans = pd.get_dummies(actions['side'], prefix='side')
    actions = pd.concat([actions, side_booleans], axis=1)

    # Create boolean variables for 'area's unique values
    area_booleans = pd.get_dummies(actions['area'], prefix='area')
    actions = pd.concat([actions, area_booleans], axis=1)

    # Create boolean variables for 'areaDetail's unique values
    areaDetail_booleans = pd.get_dummies(actions['areaDetail'], prefix='areaDetail')
    actions = pd.concat([actions, areaDetail_booleans], axis=1)

    # Create boolean variables for 'shotResult's unique values
    shotResult_booleans = pd.get_dummies(actions['shotResult'], prefix='shotResult')
    actions = pd.concat([actions, shotResult_booleans], axis=1)

    # Keep specified numeric variables and the new 'is_team_action' boolean
    numeric_columns = ['period', 'x', 'y', 'xLegacy', 'yLegacy', 'isFieldGoal', 'shotDistance',
                       'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal',
                       'reboundOffensiveTotal', 'pointsTotal', 'assistTotal', 'turnoverTotal',
                       'foulPersonalTotal', 'foulTechnicalTotal', 'clock_seconds', 'is_team_action']

    # Determine columns to keep
    columns_to_keep = numeric_columns + list(action_types.columns) + \
                      [f'subtype_{subtype}' for subtype in subtypes_of_interest] + \
                      [f'qualifier_{qualifier}' for qualifier in qualifiers_of_interest] + \
                      list(side_booleans.columns) + list(area_booleans.columns) + \
                      list(areaDetail_booleans.columns) + list(shotResult_booleans.columns)

    # Return the transformed DataFrame
    return actions[columns_to_keep]

# Example usage:
# team_id = games.loc[games['GAME_ID'] == game_id, 'teamId'].iloc[0]
# actions_final = transform_actions(final_plays_df, team_id)


In [63]:
games_data = []

# Iterate through the first 25 GAME_IDs in the games DataFrame
for game_id in games['GAME_ID'][:25]:
    print(game_id)
    # Get the teamId for the current game
    team_id = games.loc[games['GAME_ID'] == game_id, 'TEAM_ID'].iloc[0]
    
    # Instantiate PlayByPlay for the current game
    pbp = playbyplay.PlayByPlay(game_id=game_id)
    
    # Fetch the plays data
    plays = pbp.get_dict()['game']['actions']
    
    # Create a DataFrame from the plays data
    plays_df = pd.DataFrame(plays)
    
    # Transform the plays DataFrame using the team_id
    actions_final = transform_actions(plays_df, team_id)
    
    # Summarize the transformed features for the game
    # For the sake of this example, let's just count the number of actions
    game_summary = {'GAME_ID': game_id, 'actions_count': len(actions_final)}
    
    # Append the game summary to the list
    games_data.append(game_summary)

# Create a DataFrame from the games data
games_features_df = pd.DataFrame(games_data)

# Map 'W' to 1 and 'L' to 0 in the 'WL' column of the games DataFrame
games['WL_binary'] = games['WL'].map({'W': 1, 'L': 0})

# Merge the features DataFrame with the games DataFrame to include the win/loss outcome
final_df = games_features_df.merge(games[['GAME_ID', 'WL_binary']], on='GAME_ID', how='left')
final_df

0022301164
0022301161
0022301161
0022301163
0022301160
0022301165
0022301158
0022301158
0022301163
0022301159
0022301162
0022301165
0022301164
0022301162
0022301159
0022301160
0022301151
0022301146
0022301145
0022301147
0022301148
0022301145
0022301152
0022301154
2042300401


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [64]:
pbp = playbyplay.PlayByPlay(game_id='2042300401')

plays = pbp.get_dict()['game']['actions']
print(plays[5])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [66]:
games[:25]

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22023,1610612750,MIN,Minnesota Timberwolves,0022301164,2024-04-10,MIN @ DEN,None,180,80,31,59,0.525,9,19,0.474,9,16,0.563,5,22,27,17,7,5,7,12,-3.0
1,22023,1610612742,DAL,Dallas Mavericks,0022301161,2024-04-10,DAL @ MIA,W,241,111,43,87,0.494,15,44,0.341,10,13,0.769,8,31,39,27,8,8,7,13,19.8
2,22023,1610612748,MIA,Miami Heat,0022301161,2024-04-10,MIA vs. DAL,L,239,92,33,81,0.407,13,30,0.433,13,13,1.000,12,34,46,22,3,1,15,16,-18.8
3,22023,1610612759,SAS,San Antonio Spurs,0022301163,2024-04-10,SAS @ OKC,L,241,89,32,96,0.333,9,40,0.225,16,20,0.800,4,43,47,24,8,4,15,14,-38.0
4,22023,1610612761,TOR,Toronto Raptors,0022301160,2024-04-10,TOR @ BKN,L,241,102,35,94,0.372,16,40,0.400,16,18,0.889,15,30,45,19,8,5,9,22,-4.0
5,22023,1610612746,LAC,LA Clippers,0022301165,2024-04-10,LAC vs. PHX,None,120,71,29,56,0.518,8,18,0.444,5,7,0.714,3,25,28,13,4,1,6,8,1.0
6,22023,1610612739,CLE,Cleveland Cavaliers,0022301158,2024-04-10,CLE vs. MEM,W,243,110,37,82,0.451,15,37,0.405,21,29,0.724,12,35,47,27,11,10,13,15,12.0
7,22023,1610612763,MEM,Memphis Grizzlies,0022301158,2024-04-10,MEM @ CLE,L,240,98,33,84,0.393,14,32,0.438,18,19,0.947,10,34,44,21,10,10,13,21,-12.0
8,22023,1610612760,OKC,Oklahoma City Thunder,0022301163,2024-04-10,OKC vs. SAS,W,241,127,47,101,0.465,16,38,0.421,17,21,0.810,7,55,62,33,12,6,12,16,38.0
9,22023,1610612766,CHA,Charlotte Hornets,0022301159,2024-04-10,CHA @ ATL,W,241,115,40,79,0.506,14,36,0.389,21,22,0.955,5,28,33,25,11,2,13,18,1.0


In [83]:
winprobabilitypbp.WinProbabilityPBP('0022301164').get_data_frames()[0][:1500]

,GAME_ID,EVENT_NUM,HOME_PCT,VISITOR_PCT,HOME_PTS,VISITOR_PTS,HOME_SCORE_MARGIN,PERIOD,SECONDS_REMAINING,HOME_POSS_IND,HOME_G,DESCRIPTION,LOCATION,PCTIMESTRING,ISVISIBLE
0,0022301164,2.0,0.58808,0.41192,0,0,0,1,720.0,NaN,None,None,None,None,None
1,0022301164,4.0,0.58808,0.41192,0,0,0,1,720.0,0.0,None,None,None,None,None
2,0022301164,NaN,0.58806,0.41194,0,0,0,1,719.0,0.0,None,None,None,None,None
3,0022301164,NaN,0.58804,0.41196,0,0,0,1,718.0,0.0,None,None,None,None,None
4,0022301164,NaN,0.58801,0.41199,0,0,0,1,717.0,0.0,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,0022301164,NaN,0.46715,0.53285,49,52,-3,3,717.0,1.0,None,None,None,None,None
1496,0022301164,NaN,0.46708,0.53292,49,52,-3,3,716.0,1.0,None,None,None,None,None
1497,0022301164,NaN,0.46700,0.53300,49,52,-3,3,715.0,1.0,None,None,None,None,None
1498,0022301164,NaN,0.46693,0.53307,49,52,-3,3,714.0,1.0,None,None,None,None,None
